### ONNX 변환과 검증
- 파일 : n1-4_onnx.ipynb
- 역할 : PyTorch 모델을 ONNX로 내보내고, 변환 모델이 원본과 같은 답을 내는지 수치로 검증합니다.
- 기능 :
    1. 실습 패키지 설치  
    2. dummy input 준비  
    3. Tracing의 한계 확인: 입력에 따라 분기하는 모델  
    4. MobileNetV2의 ONNX 변환  
    5. 비교 모델 ResNet-18의 변환  
    6. 그래프 유효성 검사와 입출력 규격 확인  
    7. 그래프 구성 요소 집계  
    8. PyTorch와 ONNX Runtime의 출력 비교  
    9. PyTorch와 ONNX Runtime의 Latency 비교  


>> [준비] 측정 함수 준비 (노트북 전용 셀)

In [ ]:
#- 추론 파이프라인 실습에서 만든 measure()를 이 세션에도 정의
import time
import numpy as np

def measure(fn, n_warmup=10, n_runs=100):
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return t.mean(), np.median(t), np.percentile(t, 95), t.max()

# 확인 포인트 ------------------------------------------
# - 함수 정의 셀: 출력 없음

>> [코드 3-1] 실습 패키지 설치 (n1-4_onnx.ipynb, STEP 1)
<hr>

In [ ]:
#- 실습 패키지 설치와 버전 확인
!pip install -q onnx onnxruntime "protobuf<6"

#- 모듈 로딩
import onnx, onnxruntime

#- 버전 출력
print("onnx", onnx.__version__, "/ onnxruntime", onnxruntime.__version__)

# 확인 포인트 -------------------------------------------------------------------------
# - 두 패키지의 버전 출력, ERROR 문구 없음 체크

>> [코드 3-2] dummy input 준비 (STEP 2)
<hr>

In [ ]:
#- 모듈로딩
import torch
from torchvision import models

#- 변환할 원본 모델 로딩 (추론 모드)
W_MNET = models.MobileNet_V2_Weights.IMAGENET1K_V1
MNET = models.mobilenet_v2(weights=W_MNET).eval()

#- 변환용 dummy input: 값이 아니라 shape와 dtype이 전달됨
dummy = torch.randn(1, 3, 224, 224)
print(f"dummy input shape: {tuple(dummy.shape)}")

# 확인 포인트 -------------------------------------------------------------------------
# - dummy input shape (1, 3, 224, 224) 출력 체크

**실행 결과 예시**
```
dummy input shape: (1, 3, 224, 224)
```

>> [코드 3-3] Tracing의 한계 확인: 입력에 따라 분기하는 모델 (STEP 3)
<hr>

In [ ]:
#- 모듈로딩
import numpy as np, onnxruntime as ort

#- 파일 상수: 분기 실험용 변환 산출물
BRANCHY_ONNX = "branchy.onnx"

#- 입력 값에 따라 분기하는 모델 정의
class Branchy(torch.nn.Module):
    def forward(self, x):
        if x.mean() > 0:          # 입력 값에 따라 달라지는 분기
            return x * 2
        return x - 1

#- 양수 입력으로 추적해 변환: if의 한쪽 경로만 그래프에 남음
b = Branchy()
torch.onnx.export(b, torch.ones(1, 3), BRANCHY_ONNX,   # 양수 입력으로 추적
                  input_names=["x"], output_names=["y"], dynamo=False)

#- 음수 입력으로 PyTorch와 ONNX 비교
s = ort.InferenceSession(BRANCHY_ONNX, providers=["CPUExecutionProvider"])
neg = np.full((1, 3), -3.0, dtype=np.float32)
print("PyTorch :", b(torch.from_numpy(neg)).numpy())    # x - 1 경로 → -4
print("ONNX    :", s.run(None, {"x": neg})[0])          # 추적된 x * 2 경로 → -6

# 확인 포인트 -------------------------------------------------------------------------
# - 두 결과가 다르면 변환 그래프에 한쪽 분기만 남았다는 뜻
# - TracerWarning 출력은 정상 (Tracing 한계를 알리는 경고)
# - legacy export DeprecationWarning 무시 가능
# - dynamo=False 사용에 따른 안내, 이후 export 셀에서도 동일

>> [코드 3-4] MobileNetV2의 ONNX 변환 (STEP 4)
<hr>

In [ ]:
#- 모듈 로딩
import os

#- 사용할 Opset (프로젝트 내내 고정)
OPSET = 13                                     # DFC 3.34.0에서 변환을 확인한 값

#- 파일 상수: 변환 산출물
MNET_ONNX = "mobilenet_v2.onnx"                # 이후 검증·경량화 단계에서 사용

#-  한 번 실행해 계산 과정을 기록하는(Tracing) 방식으로 ONNX 변환
torch.onnx.export(
    MNET, dummy, MNET_ONNX,                         # 모델, 추적용 입력, 저장 파일
    opset_version=OPSET,                            # 연산자 사전 버전
    input_names=["input"],                          # 그래프 입력 이름
    output_names=["output"],                        # 그래프 출력 이름
    dynamo=False,                                   # 기존 Tracing 방식으로 변환
)

#- 산출물 크기 확인
print(f"파일 크기: {os.path.getsize(MNET_ONNX)/1e6:.1f} MB")
# 확인 포인트 -------------------------------------------------------------------------
# - 파일 생성과 크기(MB) 출력 체크

**실행 결과 예시**
```
파일 크기: 14.0 MB
```

>> [코드 3-5] 비교 모델 ResNet-18의 변환 (STEP 5)
<hr>

In [ ]:
#- 파일 상수: 비교 모델 변환 산출물
RNET_ONNX = "resnet18.onnx"

#- 비교 모델 ResNet-18 로딩
W_RES = models.ResNet18_Weights.IMAGENET1K_V1
RNET = models.resnet18(weights=W_RES).eval()

#- 비교 모델 ResNet-18을 같은 방식으로 변환
torch.onnx.export(
    RNET, dummy, RNET_ONNX,                         # 모델, 추적용 입력, 저장 파일
    opset_version=OPSET,                            # 연산자 사전 버전
    input_names=["input"],                          # 그래프 입력 이름
    output_names=["output"],                        # 그래프 출력 이름
    dynamo=False,                                   # 기존 Tracing 방식으로 변환
)

#- 산출물 크기 확인
print(f"파일 크기: {os.path.getsize(RNET_ONNX)/1e6:.1f} MB")

# 확인 포인트 -------------------------------------------------------------------------
# - MobileNetV2보다 큰 파일 크기 체크 (파라미터 수 차이)

**실행 결과 예시**
```
파일 크기: 46.8 MB
```

>> [코드 3-6] 그래프 유효성 검사와 입출력 규격 확인 (STEP 6)
<hr>

In [ ]:
#- 모듈로딩
import onnx

#- 파일 상수: 변환 산출물 (이후 검증·경량화 단계가 사용)
MNET_ONNX = "mobilenet_v2.onnx"

#- 그래프 로딩과 유효성 검사
m = onnx.load( MNET_ONNX )
onnx.checker.check_model(m)                                 # 그래프 유효성 검사
print("모델 구조 검사 통과")

#- 입력·출력 이름과 shape, Opset 확인
for i in m.graph.input:
    dims = [d.dim_value for d in i.type.tensor_type.shape.dim]
    print(f"Input  '{i.name}': {dims}")

for o in m.graph.output:
    dims = [d.dim_value for d in o.type.tensor_type.shape.dim]
    print(f"Output '{o.name}': {dims}")

print(f"Opset: {m.opset_import[0].version}")

# 확인 포인트 -------------------------------------------------------------------------
# - 입력과 출력 shape 체크

**실행 결과 예시**
```
모델 구조 검사 통과
Input  'input': [1, 3, 224, 224]
Output 'output': [1, 1000]
Opset: 13
```

>> [코드 3-7] 그래프 구성 요소 집계 (STEP 7)
<hr>

In [ ]:
#- 연산자 종류별 개수 집계
from collections import Counter

#- [('항목', 개수)] 형태로 가장 자주 등장하는 항목 순서대로 정렬
ops = Counter(n.op_type for n in m.graph.node)
for op, cnt in ops.most_common():
    print(f"{op:20s} {cnt}")

#- 가중치 텐서(initializer) 수 확인
print(f"initializer 수: {len(m.graph.initializer)}")

# 확인 포인트 ------------------------------------------------------------------------
# - Conv 최다, BatchNormalization 없음 체크

>> [코드 3-8] PyTorch와 ONNX Runtime의 출력 비교 (STEP 8)
<hr>

In [ ]:
#- 모듈로딩
import numpy as np
import onnxruntime as ort

#- 파일 상수: 변환 산출물 (이후 검증·경량화 단계가 사용)
MNET_ONNX = "mobilenet_v2.onnx"

#- 같은 입력 하나 준비
x = torch.randn(1, 3, 224, 224)

#- 원본 PyTorch 출력
with torch.no_grad():
    out_torch = MNET(x).numpy()

#- ONNX Runtime 출력
sess = ort.InferenceSession(MNET_ONNX,
                            providers=["CPUExecutionProvider"])
out_onnx = sess.run(None, {"input": x.numpy()})[0]

#- 수치 동등성: 최대 오차와 Top-1 일치
max_diff = np.abs(out_torch - out_onnx).max()
print(f"최대 절대 오차: {max_diff:.2e}")
print(f"Top-1 클래스 일치: {out_torch.argmax() == out_onnx.argmax()}")

# 확인 포인트 -------------------------------------------------------------------------
# - 오차 1e-5 수준(1e-4 미만), Top-1 일치 True 체크

>> [코드 3-9] PyTorch와 ONNX Runtime의 Latency 비교 (STEP 9)
<hr>

In [ ]:
#- 동일 입력으로 두 실행기의 Latency 비교 ([준비] 셀의 measure 사용)
x_np = x.numpy()

#- 추론 파이프라인에서 만든 측정 함수 measure(). 실행기에 따른 속도차 측정
with torch.inference_mode():
     lat_torch = measure(lambda: MNET(x))[0]
lat_onnx  = measure(lambda: sess.run(None, {"input": x_np}))[0]

print(f"PyTorch      : {lat_torch:7.2f} ms")
print(f"ONNX Runtime : {lat_onnx:7.2f} ms")

# 확인 포인트 -------------------------------------------------------------------------
# - 두 실행기의 Latency 출력 체크 (절댓값은 머신마다 다름)